# Parquet Basics — 01: Reading Parquet Files

## What you will learn
Parquet is the de-facto standard for analytical storage in the Spark ecosystem.
Unlike CSV, Parquet is a **binary columnar format** with built-in compression
and statistics that Spark can exploit for dramatic performance gains.

In this notebook:
1. Why Parquet is different from CSV (columnar vs row-based)
2. Basic `spark.read.parquet()` — single file, directory, glob
3. Reading with schema — explicit vs inferred
4. `mergeSchema` — reading files with different schemas
5. Parquet metadata — what Spark reads before touching data
6. The footer and why it matters for performance


In [1]:
import os, time, pathlib, shutil, random, datetime, subprocess
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/parquet_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('parquet-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | DATA_DIR: {DATA_DIR}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/09 09:19:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2 | DATA_DIR: /workspace/data/parquet_basics


## Step 1 — Columnar vs Row-Based: The Core Difference

Before reading Parquet, understand WHY it exists.

```
CSV (row-based):
  Row 1: [order_id=1, category=Electronics, revenue=999.99, country=US, ...]
  Row 2: [order_id=2, category=Books,       revenue= 49.99, country=UK, ...]
  → To get "total revenue", read EVERY byte of EVERY row

Parquet (columnar):
  revenue column: [999.99, 49.99, 349.99, ...]  ← all revenue values together
  category column:[Electronics, Books, Furniture, ...]
  → To get "total revenue", read ONLY the revenue column bytes
  → 10 columns in table but you need 1 → read 10% of the data
```
This is called **column pruning** and is Parquet's biggest advantage.


In [2]:
import os, time, pathlib, shutil, random, datetime, subprocess
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/parquet_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('parquet-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | DATA_DIR: {DATA_DIR}")

random.seed(42)
N = 200_000
CATEGORIES = ["Electronics","Clothing","Books","Food","Sports","Furniture"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA","AU","BR"]
STATUSES   = ["pending","confirmed","shipped","delivered","cancelled"]

schema = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("category",    StringType()),
    StructField("country",     StringType()),
    StructField("quantity",    IntegerType()),
    StructField("price",       DoubleType()),
    StructField("discount",    DoubleType()),
    StructField("revenue",     DoubleType()),
    StructField("status",      StringType()),
    StructField("is_returned", BooleanType()),
    StructField("order_date",  DateType()),
])

base = datetime.date(2023, 1, 1)
rows = []
for i in range(N):
    qty  = random.randint(1, 20)
    price = round(random.uniform(5, 2000), 2)
    disc  = round(random.uniform(0, 0.4), 3)
    rev   = round(qty * price * (1 - disc), 2)
    d     = base + datetime.timedelta(days=random.randint(0, 365))
    rows.append((i+1, random.randint(1,50000),
                 f"Product_{random.randint(1,500)}",
                 random.choice(CATEGORIES), random.choice(COUNTRIES),
                 qty, price, disc, rev,
                 random.choice(STATUSES), random.random() < 0.05, d))

df = spark.createDataFrame(rows, schema)
df.cache().count()
print(f"Dataset: {N:,} rows | {len(schema)} columns")
df.printSchema()

Spark 4.0.2 | DATA_DIR: /workspace/data/parquet_basics


26/04/09 09:19:31 WARN TaskSetManager: Stage 0 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.
26/04/09 09:19:33 WARN TaskSetManager: Stage 1 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.


Dataset: 200,000 rows | 12 columns
root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- country: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- revenue: double (nullable = true)
 |-- status: string (nullable = true)
 |-- is_returned: boolean (nullable = true)
 |-- order_date: date (nullable = true)



In [3]:
# Write test data as both CSV and Parquet for comparison
csv_path = f"{DATA_DIR}/orders.csv"
pq_path  = f"{DATA_DIR}/orders.parquet"

df.write.mode("overwrite").option("header", True).csv(csv_path)
df.write.mode("overwrite").parquet(pq_path)

# Size comparison
import glob as glib
csv_mb = sum(pathlib.Path(f).stat().st_size for f in glib.glob(f"{csv_path}/*.csv")) / 1024/1024
pq_mb  = sum(pathlib.Path(f).stat().st_size for f in glib.glob(f"{pq_path}/*.parquet")) / 1024/1024

print(f"Storage comparison ({N:,} rows):")
print(f"  CSV     : {csv_mb:.1f} MB")
print(f"  Parquet : {pq_mb:.1f} MB  ({csv_mb/pq_mb:.1f}x smaller)")

26/04/09 09:19:34 WARN TaskSetManager: Stage 4 contains a task of very large size (3126 KiB). The maximum recommended task size is 1000 KiB.
26/04/09 09:19:35 WARN TaskSetManager: Stage 5 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

Storage comparison (200,000 rows):
  CSV     : 16.3 MB
  Parquet : 4.6 MB  (3.5x smaller)


## Step 2 — Basic Read: Single File, Directory, Glob


In [4]:
# 1. Read from directory (most common — contains multiple part files)
df_dir = spark.read.parquet(pq_path)
print(f"1. Read directory: {df_dir.count():,} rows")
df_dir.printSchema()
df_dir.show(3)

1. Read directory: 200,000 rows
root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- country: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- revenue: double (nullable = true)
 |-- status: string (nullable = true)
 |-- is_returned: boolean (nullable = true)
 |-- order_date: date (nullable = true)

+--------+-----------+-----------+-----------+-------+--------+-------+--------+--------+---------+-----------+----------+
|order_id|customer_id|    product|   category|country|quantity|  price|discount| revenue|   status|is_returned|order_date|
+--------+-----------+-----------+-----------+-------+--------+-------+--------+--------+---------+-----------+----------+
|       1|       9145|Product_378|Electronics|     UK|       4|   54.9|    0.11|  195.44|cancelled|      false|2023-0

In [5]:
# 2. Read specific part file
import glob as glib
part_files = glib.glob(f"{pq_path}/*.parquet")
print(f"Part files in directory: {len(part_files)}")

df_single = spark.read.parquet(part_files[0])
print(f"\n2. Read single file: {df_single.count():,} rows")

# 3. Read with glob pattern
df_glob = spark.read.parquet(f"{pq_path}/part-0000*.parquet")
print(f"\n3. Read with glob (part-0000*): {df_glob.count():,} rows")

# 4. Read multiple paths as list
df_list = spark.read.parquet(*part_files[:2])
print(f"\n4. Read list of 2 files: {df_list.count():,} rows")

Part files in directory: 4

2. Read single file: 50,176 rows


26/04/09 09:19:38 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /workspace/data/parquet_basics/orders.parquet/part-0000*.parquet.
java.io.FileNotFoundException: File /workspace/data/parquet_basics/orders.parquet/part-0000*.parquet does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:917)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1238)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:907)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.FileStreamSink$.hasMetadata(FileStreamSink.scala:56)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:381)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSo


3. Read with glob (part-0000*): 200,000 rows

4. Read list of 2 files: 100,352 rows


## Step 3 — Schema: Inferred vs Explicit

Parquet stores the schema in its footer — unlike CSV, you don't NEED
to specify the schema. But explicit schema is still recommended for:
- Controlling nullability
- Type promotion (e.g., int → long)
- Filtering columns you don't need


In [6]:
# Inferred schema — Spark reads footer metadata (NOT the data)
t0 = time.time()
df_inferred = spark.read.parquet(pq_path)
df_inferred.count()
t_infer = time.time() - t0

print(f"Inferred schema (from Parquet footer — fast):")
df_inferred.printSchema()
print(f"Read time: {t_infer:.3f}s")

# Explicit schema — guarantees types and nullability
explicit_schema = StructType([
    StructField("order_id",    LongType(),    nullable=False),
    StructField("customer_id", LongType(),    nullable=False),
    StructField("product",     StringType(),  nullable=True),
    StructField("category",    StringType(),  nullable=True),
    StructField("country",     StringType(),  nullable=True),
    StructField("quantity",    IntegerType(), nullable=True),
    StructField("price",       DoubleType(),  nullable=True),
    StructField("discount",    DoubleType(),  nullable=True),
    StructField("revenue",     DoubleType(),  nullable=True),
    StructField("status",      StringType(),  nullable=True),
    StructField("is_returned", BooleanType(), nullable=True),
    StructField("order_date",  DateType(),    nullable=True),
])

t0 = time.time()
df_explicit = spark.read.schema(explicit_schema).parquet(pq_path)
df_explicit.count()
t_explicit = time.time() - t0

print(f"\nExplicit schema (same data, guaranteed types):")
df_explicit.printSchema()
print(f"Read time: {t_explicit:.3f}s")

Inferred schema (from Parquet footer — fast):
root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- country: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- revenue: double (nullable = true)
 |-- status: string (nullable = true)
 |-- is_returned: boolean (nullable = true)
 |-- order_date: date (nullable = true)

Read time: 0.334s

Explicit schema (same data, guaranteed types):
root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- country: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- revenue: double (nullable = true)
 |-- status: string (nullable = true)
 |-- is_r

## Step 4 — mergeSchema: Reading Evolved Files

When Parquet files have different schemas (column added over time),
`mergeSchema=True` combines them automatically.


In [7]:
# Version 1: original schema
v1_path = f"{DATA_DIR}/versioned/v1"
df.select("order_id","category","revenue","order_date") \
  .write.mode("overwrite").parquet(v1_path)

# Version 2: added 'discount' and 'country' columns
v2_path = f"{DATA_DIR}/versioned/v2"
df.select("order_id","category","revenue","order_date","discount","country") \
  .write.mode("overwrite").parquet(v2_path)

print("v1 schema:", spark.read.parquet(v1_path).columns)
print("v2 schema:", spark.read.parquet(v2_path).columns)

# Without mergeSchema — fails or returns only common columns
print("\nWithout mergeSchema (reads v1 schema only for all files):")
df_no_merge = spark.read.parquet(v1_path, v2_path)
print(f"  Columns: {df_no_merge.columns}")

# With mergeSchema — union of all columns
print("\nWith mergeSchema=True (union of all schemas):")
df_merged = spark.read.option("mergeSchema", True).parquet(v1_path, v2_path)
print(f"  Columns: {df_merged.columns}")
df_merged.show(4)
print("v1 rows have null for discount and country — expected")

26/04/09 09:19:39 WARN TaskSetManager: Stage 30 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.
26/04/09 09:19:40 WARN TaskSetManager: Stage 31 contains a task of very large size (3126 KiB). The maximum recommended task size is 1000 KiB.


v1 schema: ['order_id', 'category', 'revenue', 'order_date']
v2 schema: ['order_id', 'category', 'revenue', 'order_date', 'discount', 'country']

Without mergeSchema (reads v1 schema only for all files):
  Columns: ['order_id', 'category', 'revenue', 'order_date']

With mergeSchema=True (union of all schemas):
  Columns: ['order_id', 'category', 'revenue', 'order_date', 'discount', 'country']
+--------+-----------+--------+----------+--------+-------+
|order_id|   category| revenue|order_date|discount|country|
+--------+-----------+--------+----------+--------+-------+
|       1|Electronics|  195.44|2023-04-25|    0.11|     UK|
|       2|   Clothing|  174.07|2023-11-05|   0.093|     AU|
|       3|      Books|14534.56|2023-03-23|   0.003|     JP|
|       4|      Books| 1948.71|2023-07-03|   0.152|     US|
+--------+-----------+--------+----------+--------+-------+
only showing top 4 rows
v1 rows have null for discount and country — expected


## Step 5 — Parquet Footer and Metadata

Parquet stores rich metadata in the **file footer**:
- Schema definition
- Row group locations
- Column statistics (min/max/null count per row group)

Spark reads ONLY the footer first — this is why Parquet schema inference
is nearly free (no data scan needed).


In [8]:
# Inspect Parquet metadata via pyarrow
try:
    import pyarrow.parquet as pq
    import glob as glib

    pq_file = glib.glob(f"{pq_path}/*.parquet")[0]
    meta = pq.ParquetFile(pq_file).metadata

    print(f"Parquet file metadata:")
    print(f"  File               : {pq_file.split('/')[-1]}")
    print(f"  Rows in this file  : {meta.num_rows:,}")
    print(f"  Columns            : {meta.num_columns}")
    print(f"  Row groups         : {meta.num_row_groups}")
    print(f"  Created by         : {meta.created_by[:60]}")
    print()

    rg = meta.row_group(0)
    print(f"Row Group 0:")
    print(f"  Rows     : {rg.num_rows:,}")
    print(f"  Size     : {rg.total_byte_size/1024/1024:.2f} MB")
    print()
    print(f"  {'Column':<20} {'Min':>20} {'Max':>20} {'Nulls':>8}")
    print("  " + "-"*72)
    for ci in range(min(6, meta.num_columns)):
        col   = rg.column(ci)
        stats = col.statistics
        if stats and stats.has_min_max:
            mn = str(stats.min)[:18]
            mx = str(stats.max)[:18]
        else:
            mn = mx = "N/A"
        nulls = stats.null_count if stats else "N/A"
        print(f"  {col.path_in_schema:<20} {mn:>20} {mx:>20} {nulls:>8}")

    print()
    print("These min/max stats are used for DATA SKIPPING:")
    print("  Query: WHERE revenue > 10000")
    print("  Spark checks: max(revenue) in each row group")
    print("  If max < 10000 → skip entire row group (no I/O!)")
except ImportError:
    print("pyarrow not available — metadata shown via Spark explain:")
    spark.read.parquet(pq_path).filter(F.col("revenue") > 10000).explain(mode="formatted")

Parquet file metadata:
  File               : part-00000-18eb657e-790f-4040-82a5-b3cf67ec030a-c000.snappy.parquet
  Rows in this file  : 50,176
  Columns            : 12
  Row groups         : 1
  Created by         : parquet-mr version 1.15.2 (build 859eac165b08f927fa14590c33b

Row Group 0:
  Rows     : 50,176
  Size     : 1.80 MB

  Column                                Min                  Max    Nulls
  ------------------------------------------------------------------------
  order_id                                1                50176        0
  customer_id                             1                50000        0
  product                         Product_1           Product_99        0
  category                            Books               Sports        0
  country                                AU                   US        0
  quantity                                1                   20        0

These min/max stats are used for DATA SKIPPING:
  Query: WHERE revenue 

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 59732)
Traceback (most recent call last):
  File "/usr/lib/python3.10/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/usr/lib/python3.10/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/usr/lib/python3.10/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/usr/lib/python3.10/socketserver.py", line 747, in __init__
    self.handle()
  File "/opt/spark/python/pyspark/accumulators.py", line 299, in handle
    poll(accum_updates)
  File "/opt/spark/python/pyspark/accumulators.py", line 271, in poll
    if self.rfile in r and func():
  File "/opt/spark/python/pyspark/accumulators.py", line 275, in accum_updates
    num_updates = read_int(self.rfile)
  File "/opt/spark/python/pyspark/serializers.py", lin

## Summary: Parquet Read Cheat Sheet

```python
# Basic read
spark.read.parquet("/path/to/dir")
spark.read.parquet("/path/to/dir", "/another/path")   # multiple paths
spark.read.parquet("/path/to/part-0001*.parquet")      # glob

# With options
spark.read
     .option("mergeSchema", True)     # union schemas from multiple files
     .option("recursiveFileLookup", True)  # scan subdirectories too
     .parquet("/path/")

# With explicit schema
spark.read.schema(my_schema).parquet("/path/")

# Read only specific columns (column pruning — much faster)
spark.read.parquet("/path/").select("revenue", "category")
```

**Key advantage over CSV:**
- Schema stored in footer → no schema inference scan
- Column statistics → data skipping without reading data
- Columnar layout → read only columns you need
- Compressed → less I/O

**Next:** `02_writing_parquet.ipynb` — controlling output format, compression, partitioning.
